# Four Breaths of Mumbai — Google Colab evidence build

This notebook uses Google Colab to audit the cached air-quality observations and reproduce the final one-page HTML. Upload `colab_bundle.zip` from `project_submit`. The downloaded page is static and works offline; Colab is a build tool, not a page dependency.


In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile

uploaded = files.upload()
assert 'colab_bundle.zip' in uploaded, 'Upload colab_bundle.zip from project_submit'
with ZipFile('colab_bundle.zip') as archive:
    archive.extractall('/content')
project = Path('/content/project_submit')
print(f'Cached project extracted to {project}')


## Audit the signature comparison

At each complete hour, compare one station with the median of the other three. This retains the same weather and time context and avoids filling missing readings. The published night statistic is computed from matched hours, not estimated from the chart.


In [ ]:
import pandas as pd

obs = pd.read_csv(project / 'data/station_pm25_hourly.csv.gz', parse_dates=['target_start_ist'])
window = obs.loc[obs.target_start_ist.between(
    pd.Timestamp('2026-10-01', tz='Asia/Kolkata'),
    pd.Timestamp('2026-11-07', tz='Asia/Kolkata'), inclusive='left')]
order = ['Malad West', 'Shivaji Nagar', 'Byculla', 'Borivali East']
matched = window.pivot(index='target_start_ist', columns='station', values='pm25')[order].dropna()
assert len(matched) == 653, 'Cached snapshot changed; review the page figures'
for station in order:
    peers = [name for name in order if name != station]
    difference = matched[station] - matched[peers].median(axis=1)
    night = (difference.index.hour >= 20) | (difference.index.hour < 8)
    print(f'{station:17} night minus peers: {difference.loc[night].median():+5.1f} µg/m³')
print(f'{len(matched)} complete four-station hours out of 888 possible')


## Generate and check the offline page

The Python builder reads only the bundled snapshots. It writes all plots as inline SVG and needs no live API call.


In [ ]:
import json
import subprocess
import sys
from zipfile import ZipFile, ZIP_DEFLATED

subprocess.run([sys.executable, str(project / 'build.py')], check=True)
metrics = json.loads((project / 'data/analysis_metrics.json').read_text())
assert metrics['peer_comparison']['matched_hours'] == len(matched)
page = (project / 'index.html').read_text()
assert '<script' not in page.lower()
assert page.count('<svg') == 7
assert '__' not in page
print(f'Offline page: {len(page):,} characters, {page.count("<svg")} inline charts')
bundle = Path('/content/four_breaths_offline_page.zip')
with ZipFile(bundle, 'w', ZIP_DEFLATED) as archive:
    archive.write(project / 'index.html', 'index.html')
    for font in sorted((project / 'assets').glob('*.woff2')):
        archive.write(font, f'assets/{font.name}')
files.download(str(bundle))
